# Illusion Diffusion
By [Matt Tancik](https://www.matthewtancik.com/)

Based on fast.ai [diffusion notebooks](https://github.com/fastai/diffusion-nbs).

![ChessUrl](https://user-images.githubusercontent.com/3310961/218217058-4c3daad4-1e3f-4f14-8c50-782d40c70501.gif "chess")



In [1]:
from google.colab import drive
drive.mount('/content/drive')
FOLDERNAME = "Visual_Anagrams_2024CVPR/baselines/tancik"
assert FOLDERNAME is not None, "[!] Enter the foldername."
import sys
sys.path.append('/content/drive/MyDrive/{}'.format(FOLDERNAME))

Mounted at /content/drive


In [ ]:
from huggingface_hub import login
token = ""
login(token=token)

In [5]:
# @title #Install requirements
!pip install -q --upgrade transformers diffusers ftfy mediapy

from base64 import b64encode

import torch
from diffusers import AutoencoderKL, LMSDiscreteScheduler, UNet2DConditionModel
from huggingface_hub import notebook_login

import numpy as np
import mediapy as mp
import cv2

from pathlib import Path
from torch import autocast
from tqdm.auto import tqdm
from transformers import CLIPTextModel, CLIPTokenizer, logging

torch.manual_seed(1)
# if not (Path.home()/'.huggingface'/'token').exists(): notebook_login()

# Supress some unnecessary warnings when loading the CLIPTextModel
logging.set_verbosity_error()

# Set device
torch_device = "cuda" if torch.cuda.is_available() else "cpu"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 84.1 MB/s eta 0:00:00


In [6]:
# @title #Load Models
# Load the autoencoder model which will be used to decode the latents into image space.
vae = AutoencoderKL.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="vae", torch_dtype=torch.float16)

# Load the tokenizer and text encoder to tokenize and encode the text.
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14")

# The UNet model for generating the latents.
unet = UNet2DConditionModel.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="unet", torch_dtype=torch.float16)

# The noise scheduler
scheduler = LMSDiscreteScheduler(beta_start=0.00085, beta_end=0.012, beta_schedule="scaled_linear", num_train_timesteps=1000)

# To the GPU we go!
vae = vae.to(torch_device)
text_encoder = text_encoder.to(torch_device)
unet = unet.to(torch_device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/961k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

In [7]:
output_folder = "/content/drive/My Drive/Visual_Anagrams_2024CVPR/baselines/tancik/outputs"
illusion_configs = [
    {
        "name": "flip.campfire.man",
        "prompts": ["an oil painting of people around a campfire", "an oil painting of an old man"],
        "views": ["identity", "flip"],
    },
    {
        "name": "jigsaw.houseplants.marilyn",
        "prompts": ["an oil painting of houseplants", "an oil painting of marilyn monroe"],
        "views": ["identity", "jigsaw"],
    },
    {
        "name": "inner.einstein.marilyn",
        "prompts": ["an oil painting of albert einstein", "an oil painting of marilyn monroe"],
        "views": ["identity", "inner_circle"],
    },
    {
        "name": "negate.landscape.houseplants",
        "prompts": ["a lithograph of a landscape", "a lithograph of houseplants"],
        "views": ["identity", "negate"],
    },
    {
        "name": "patch.lemur.kangaroo",
        "prompts": ["a pencil sketch of a lemur", "a pencil sketch of a kangaroo"],
        "views": ["identity", "patch_permute"],
    },
    {
        "name": "pixel.duck.rabbit",
        "prompts": ["a mosaic of a duck", "a mosaic of a rabbit"],
        "views": ["identity", "pixel_permute"],
    },
    {
        "name": "skew.tudor.skull",
        "prompts": ["an oil painting of a tudor portrait", "an oil painting of a skull"],
        "views": ["identity", "skew"],
    },
    {
        "name": "skew.taylor.rose",
        "prompts": ["an oil painting of a Taylor Swift", "an oil painting of a rose"],
        "views": ["identity", "skew"],
    },
]

# Function Pipeline For View 2
Generate 3 different size images in one task.
Specifically, 64, 256, 1024.

In [10]:
# Function Version
from PIL import Image
import torchvision.transforms.functional as TF
from datetime import datetime
import os
import cv2
import numpy as np

def create_folder(base_output_folder, name):
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    folder = f"{base_output_folder}/{name}_{timestamp}"
    os.makedirs(folder, exist_ok=True)
    return folder, timestamp

def save_image(image_tensor, prompt_1, prompt_2, resolution, timestamp, folder):
    name1 = prompt_1.replace(" ", "_")[:30]
    name2 = prompt_2.replace(" ", "_")[:30]
    filename = f"{timestamp}_{name1}_to_{name2}_{resolution}.png"
    path = os.path.join(folder, filename)
    pil_img = TF.to_pil_image(image_tensor[0] / 2. + 0.5)
    pil_img.save(path)
    print(f"✅ Saved image: {path}")
    pil_img.show()

def generate_illusion_animation(image_tensor, degrees=90, num_frames=300):
    def rotate_image(image, angle):
        rows, cols, _ = image.shape
        rot_mat = cv2.getRotationMatrix2D((cols / 2, rows / 2), angle, 1)
        return cv2.warpAffine(image, rot_mat, (cols, rows), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)

    def easeInOutQuint(x):
        if x < 0.5:
            return 4 * x**3
        else:
            return 1 - (-2 * x + 2)**3 / 2

    # Convert tensor to image (uint8 ndarray)
    image = image_tensor[0].detach().cpu()
    image = (image / 2 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()
    image = (image * 255).round().astype("uint8")

    # Generate ease-in-out rotation values
    ts = np.concatenate([
        np.zeros(num_frames // 4),
        np.linspace(0, 1, num_frames // 4),
        np.ones(num_frames // 4),
        np.linspace(1, 0, num_frames // 4),
    ])
    ts = [easeInOutQuint(x) for x in ts]

    # Generate rotated frames
    frames = [rotate_image(image, t * degrees) for t in ts]

    # Display video
    mp.show_video(frames)


def generate_multi_res_illusion_images(prompt1, prompt2, name, base_output_folder,
                                       resolutions=[1024],
                                       rotation_deg=180,
                                       num_inference_steps=500,
                                       guidance_scale=11.0,
                                       seed=25,
                                       batch_size=1):

    rotate = int(rotation_deg) // 90
    torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    generator = torch.manual_seed(seed)

    # Prepare output folder
    output_dir, timestamp = create_folder(base_output_folder, name)

    # Prompt embedding (once for all resolutions)
    text_input_1 = tokenizer([prompt1], padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")
    text_input_2 = tokenizer([prompt2], padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")
    with torch.no_grad():
        emb_1 = text_encoder(text_input_1.input_ids.to(torch_device))[0]
        emb_2 = text_encoder(text_input_2.input_ids.to(torch_device))[0]
    uncond_input = tokenizer([""] * batch_size, padding="max_length", max_length=text_input_1.input_ids.shape[-1], return_tensors="pt")
    with torch.no_grad():
        uncond_emb = text_encoder(uncond_input.input_ids.to(torch_device))[0]
    text_embeddings_1 = torch.cat([uncond_emb, emb_1])
    text_embeddings_2 = torch.cat([uncond_emb, emb_2])

    for res in resolutions:
        print(f"\n🔍 Generating resolution: {res}x{res}")
        height = width = res

        # Prep scheduler
        scheduler.set_timesteps(num_inference_steps)

        # Init latent
        latents = torch.randn((batch_size, unet.in_channels, height // 8, width // 8), generator=generator).half().to(torch_device)
        latents *= scheduler.init_noise_sigma

        with autocast("cuda"):
            for i, t in tqdm(enumerate(scheduler.timesteps), total=len(scheduler.timesteps)):
                if i % 2 == 0:
                    latents = torch.rot90(latents, rotate, [2, 3])
                    latent_input = torch.cat([latents] * 2)
                else:
                    latent_input = torch.cat([latents] * 2)

                latent_input = scheduler.scale_model_input(latent_input, t)

                with torch.no_grad():
                    noise_pred = unet(latent_input, t, encoder_hidden_states=(text_embeddings_1 if i % 2 == 0 else text_embeddings_2)).sample

                noise_uncond, noise_text = noise_pred.chunk(2)
                guided_noise = noise_uncond + guidance_scale * (noise_text - noise_uncond)

                if i % 2 == 0:
                    guided_noise = torch.rot90(guided_noise, -rotate, [2, 3])
                    latents = torch.rot90(latents, -rotate, [2, 3])

                latents = scheduler.step(guided_noise, t, latents).prev_sample

        # Decode
        latents = 1 / 0.18215 * latents
        with torch.no_grad():
            image = vae.decode(latents).sample

        # Save and show
        save_image(image, prompt1, prompt2, res, timestamp, output_dir)
        if res == 1024:
            generate_illusion_animation(image, degrees=rotation_deg)


## Launcher


In [11]:
for config in illusion_configs:
    generate_multi_res_illusion_images(
        prompt1=config["prompts"][0],
        prompt2=config["prompts"][1],
        name=config["name"],
        base_output_folder=output_folder,
        num_inference_steps=500
    )


Output hidden; open in https://colab.research.google.com to view.